### NLP Assignment 1 - Sentence Generator using N-gram
Team Members: ID1_ID2_ID3_ID4

In [1]:
# ============================================
# CELL 1: Imports and Common Interface
# ============================================

import nltk
from nltk.corpus import brown
import random
import string
from collections import defaultdict

# Download required NLTK data
nltk.download('brown')
nltk.download('punkt')


[nltk_data] Downloading package brown to
[nltk_data]     C:\Users\hp\AppData\Roaming\nltk_data...
[nltk_data]   Package brown is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\hp\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

COMMON INTERFACE DEFINITIONS
DO NOT MODIFY

DATA STRUCTURES AGREEMENT:
1. preprocessed_corpus: list of lists
   Each inner list is a sentence, each element is a word (string)
   Example: [["the", "cat", "sat"], ["dogs", "bark"]]

2. bigram_dict: dictionary with tuple keys and int values
   Key: (word1, word2) tuple of two words
   Value: frequency count
   Example: {("the", "cat"): 5, ("cat", "sat"): 3}

3. trigram_dict: dictionary with tuple keys and int values
   Key: (word1, word2, word3) tuple of three words
   Value: frequency count
   Example: {("the", "cat", "sat"): 2, ("cat", "sat", "on"): 1}

In [2]:
# ============================================
# CELL 2: Person 1 - UI and Input Handling (Ahmed - 20220040)
# ============================================

def get_user_input():
    """
    Person 1's function
    Returns: tuple (M, N, maxLen) as integers
    """
    while True:
        try:
            M = int(input("Enter number of sentences to generate (M): "))
            if M <= 0:
                print("M must be a positive integer. Please try again.")
                continue
            
            N = int(input("Enter n-gram order (2 for bigram, 3 for trigram): "))
            if N not in [2, 3]:
                print("N must be 2 or 3. Please try again.")
                continue
            
            maxLen = int(input("Enter maximum sentence length (maxLen): "))
            if maxLen <= 0:
                print("maxLen must be a positive integer. Please try again.")
                continue
            
            return M, N, maxLen
        
        except ValueError:
            print("Invalid input. Please enter integers only.")

def display_sentences(sentences):
    """
    Person 1's function
    Input: list of generated sentences (each sentence is a list of words)
    Output: prints sentences nicely
    """
    for i, sentence in enumerate(sentences, 1):
        # Join words into a single string
        sentence_str = ' '.join(sentence)
        print(f"Sentence {i}: {sentence_str}")

In [3]:
# ============================================
# CELL 3: Person 2 - Data Collection and Preprocessing (Sobhi - 20221088)
# ============================================

def remove_punctuation(word):
    """
    Helper function for Person 2
    Input: word string
    Returns: word string with punctuation removed
    """
    # Remove punctuation from the word
    return ''.join(char for char in word if char not in string.punctuation)

def load_and_preprocess_corpus():
    """
    Person 2's function
    Returns: preprocessed_corpus (list of sentences, each sentence is list of words)
    Requirements:
    1. Get sentences from brown corpus until reaching 200,000 tokens (±100)
    2. Tokenize each sentence into words
    3. Remove punctuation from tokens
    Note: 200,000 is limit on original corpus tokens, not vocabulary
    """
    preprocessed_corpus = []
    total_tokens = 0
    target_tokens = 200000
    tolerance = 100
    
    print("Loading Brown corpus...")
    
    # Get all sentences from brown corpus
    for sentence in brown.sents():
        if total_tokens >= target_tokens + tolerance:
            break
            
        # Process each sentence
        processed_sentence = []
        for word in sentence:
            if total_tokens >= target_tokens + tolerance:
                break
                
            # Remove punctuation and convert to lowercase
            clean_word = remove_punctuation(word.lower())
            
            # Only add non-empty words
            if clean_word:
                processed_sentence.append(clean_word)
                total_tokens += 1
        
        # Add sentence to corpus if it has at least one word
        if processed_sentence:
            preprocessed_corpus.append(processed_sentence)
    
    print(f"✓ Total tokens collected: {total_tokens}")
    print(f"✓ Total sentences collected: {len(preprocessed_corpus)}")
    
    return preprocessed_corpus

def get_vocabulary(preprocessed_corpus):
    """
    Create vocabulary set from corpus tokens
    Returns: set of unique words in the corpus
    """
    vocabulary = set()
    for sentence in preprocessed_corpus:
        vocabulary.update(sentence)
    print(f"✓ Vocabulary size: {len(vocabulary)} unique words")
    return vocabulary

In [4]:
# ============================================
# CELL 4: Person 3 - N-gram Model Building
# ============================================
def build_unigram_model(preprocessed_corpus):
    """
    NEW: Build unigram model for faster probability calculations
    Returns: dictionary of {word: count}
    """
    unigram_dict = defaultdict(int)
    for sentence in preprocessed_corpus:
        for word in sentence:
            unigram_dict[word] += 1
    return dict(unigram_dict)

def build_bigram_model(preprocessed_corpus):
    bigram_dict = defaultdict(int)
    for sentence in preprocessed_corpus:
        for i in range(len(sentence) - 1):
            bigram = (sentence[i], sentence[i+1])
            bigram_dict[bigram] += 1
    return dict(bigram_dict)

def build_trigram_model(preprocessed_corpus):
    trigram_dict = defaultdict(int)
    for sentence in preprocessed_corpus:
        for i in range(len(sentence) - 2):
            trigram = (sentence[i], sentence[i+1], sentence[i+2])
            trigram_dict[trigram] += 1
    return dict(trigram_dict)

def get_ngram_counts(preprocessed_corpus, n):
    ngram_dict = defaultdict(int)
    for sentence in preprocessed_corpus:
        for i in range(len(sentence) - n + 1):
            ngram = tuple(sentence[i:i+n])
            ngram_dict[ngram] += 1
    return dict(ngram_dict)

In [5]:
# ============================================
# CELL 5: Person 4 - Sentence Generator and Integration (Mariam - 20221217) - OPTIMIZED
# ============================================

def precompute_bigram_probabilities(bigram_dict, unigram_dict):
    """
    NEW: Precompute all bigram probabilities
    Returns: dictionary of {(prev_word, word): probability}
    """
    bigram_probs = {}
    
    for (prev_word, word), count in bigram_dict.items():
        # P(word | prev_word) = count(prev_word, word) / count(prev_word)
        prev_word_count = unigram_dict.get(prev_word, 0)
        if prev_word_count > 0:
            bigram_probs[(prev_word, word)] = count / prev_word_count
    
    return bigram_probs

def precompute_trigram_probabilities(trigram_dict, bigram_dict):
    """
    NEW: Precompute all trigram probabilities
    Returns: dictionary of {(prev_word1, prev_word2, word): probability}
    """
    trigram_probs = {}
    
    for (prev_word1, prev_word2, word), count in trigram_dict.items():
        # P(word | prev_word1, prev_word2) = count(prev_word1, prev_word2, word) / count(prev_word1, prev_word2)
        bigram_count = bigram_dict.get((prev_word1, prev_word2), 0)
        if bigram_count > 0:
            trigram_probs[(prev_word1, prev_word2, word)] = count / bigram_count
    
    return trigram_probs

def build_context_to_next_words(n, bigram_probs, trigram_probs):
    """
    NEW: Organize probabilities by context for O(1) lookup
    Returns: dictionary of {context: [(word, prob), ...]}
    """
    context_dict = {}
    
    if n == 2:
        # For bigram: context is single word
        for (prev_word, word), prob in bigram_probs.items():
            if prev_word not in context_dict:
                context_dict[prev_word] = []
            context_dict[prev_word].append((word, prob))
    
    else:  # n == 3
        # For trigram: context is word pair
        for (prev_word1, prev_word2, word), prob in trigram_probs.items():
            context = (prev_word1, prev_word2)
            if context not in context_dict:
                context_dict[context] = []
            context_dict[context].append((word, prob))
    
    # Sort each list by probability descending for quick top-3 access
    for context in context_dict:
        context_dict[context].sort(key=lambda x: x[1], reverse=True)
    
    return context_dict

def get_random_starter(n, preprocessed_corpus):
    """
    Optimized helper function
    Returns: random (n-1)gram as tuple to start sentence generation
    """
    if not preprocessed_corpus:
        return None
    
    # Pick a random sentence
    sentence = random.choice(preprocessed_corpus)
    
    if n == 2:
        # Return random single word
        if sentence:
            return (random.choice(sentence),)
    
    elif n == 3:
        # Return random word pair (need at least 2 words)
        if len(sentence) >= 2:
            idx = random.randint(0, len(sentence) - 2)
            return (sentence[idx], sentence[idx+1])
    
    # Fallback
    return None

def select_next_word_top3_optimized(current_context, n, context_dict, vocabulary):
    """
    OPTIMIZED: O(1) lookup instead of scanning all vocabulary
    Selects from top 3 words with highest probability (randomly)
    """
    # Direct lookup in precomputed context dictionary
    if n == 2:
        next_words = context_dict.get(current_context[0], [])
    else:  # n == 3
        next_words = context_dict.get(current_context, [])
    
    if len(next_words) >= 3:
        # Take top 3
        top_words = next_words[:3]
    elif next_words:
        # Take whatever we have (1 or 2 words)
        top_words = next_words
    else:
        # No words found for this context - return random from vocabulary
        return random.choice(list(vocabulary))
    
    # Select randomly from top words
    selected = random.choice(top_words)
    return selected[0]  # Return the word, not the probability

def generate_sentences_optimized(M, N, maxLen, bigram_probs, trigram_probs, 
                                    preprocessed_corpus, vocabulary, context_dict):
    """
    OPTIMIZED version of sentence generator
    """
    generated_sentences = []
    
    for sentence_num in range(M):
        # Get random starter
        starter = get_random_starter(N, preprocessed_corpus)
        
        if starter is None:
            # Fallback: start with random word from vocabulary
            starter = (random.choice(list(vocabulary)),)
        
        # Initialize sentence with starter
        if N == 2:
            sentence = [starter[0]]
            current_context = starter
        else:  # N == 3
            sentence = [starter[0], starter[1]]
            current_context = starter
        
        # Generate remaining words
        while len(sentence) < maxLen:
            # Use optimized O(1) lookup
            next_word = select_next_word_top3_optimized(
                current_context, N, context_dict, vocabulary
            )
            
            sentence.append(next_word)
            
            # Update context for next iteration
            if N == 2:
                current_context = (next_word,)
            else:  # N == 3
                current_context = (current_context[1], next_word)
        
        generated_sentences.append(sentence)
        
        # Show progress every 10 sentences
        if (sentence_num + 1) % 10 == 0:
            print(f"  Generated {sentence_num + 1}/{M} sentences...")
    
    return generated_sentences

In [6]:
# ============================================
# CELL 6: Person 4 - Main Integration Function (OPTIMIZED)
# ============================================

def main():
    """
    Person 4 - Integration of all components with optimizations
    """
    print("=" * 50)
    print("NLP Assignment 1 - Sentence Generator (OPTIMIZED)")
    print("Team: 20221217_20220136_21088_20220040")
    print("=" * 50)
    
    try:
        # Person 1's part
        print("\n[Step 1] Getting user input...")
        M, N, maxLen = get_user_input()
        print(f"✓ Generating {M} sentences with N={N}, maxLen={maxLen}")
        
        # Person 2's part
        print("\n[Step 2] Loading and preprocessing Brown corpus...")
        corpus = load_and_preprocess_corpus()
        total_tokens = sum(len(sent) for sent in corpus)
        print(f"✓ Corpus loaded with {total_tokens} tokens in {len(corpus)} sentences")
        
        # Get vocabulary from corpus
        vocabulary = get_vocabulary(corpus)
        
        # Person 3's part - Now with unigram model
        print("\n[Step 3] Building n-gram models...")
        print("  - Building unigram model...")
        unigram_dict = build_unigram_model(corpus)
        
        print("  - Building bigram model...")
        bigram_dict = build_bigram_model(corpus)
        
        print("  - Building trigram model...")
        trigram_dict = build_trigram_model(corpus)
        
        print(f"✓ Unigram model: {len(unigram_dict)} unique words")
        print(f"✓ Bigram model: {len(bigram_dict)} unique bigrams")
        print(f"✓ Trigram model: {len(trigram_dict)} unique trigrams")
        
        # Person 4's part - Precomputation
        print("\n[Step 4] Precomputing probabilities for faster generation...")
        
        print("  - Precomputing bigram probabilities...")
        bigram_probs = precompute_bigram_probabilities(bigram_dict, unigram_dict)
        
        print("  - Precomputing trigram probabilities...")
        trigram_probs = precompute_trigram_probabilities(trigram_dict, bigram_dict)
        
        print("  - Organizing by context for O(1) lookup...")
        if N == 2:
            context_dict = build_context_to_next_words(2, bigram_probs, None)
            print(f"✓ Built {len(context_dict)} context entries for bigram")
        else:
            context_dict = build_context_to_next_words(3, None, trigram_probs)
            print(f"✓ Built {len(context_dict)} context entries for trigram")
        
        # Person 4's part - Generation
        print(f"\n[Step 5] Generating {M} sentences (optimized)...")
        import time
        start_time = time.time()
        
        generated_sentences = generate_sentences_optimized(
            M, N, maxLen, bigram_probs, trigram_probs, 
            corpus, vocabulary, context_dict
        )
        
        elapsed_time = time.time() - start_time
        print(f"✓ Generated {len(generated_sentences)} sentences in {elapsed_time:.2f} seconds")
        
        # Person 1's part again
        print("\n[Step 6] Displaying generated sentences:")
        print("-" * 40)
        display_sentences(generated_sentences)
        print("-" * 40)
        
        print("\n" + "=" * 50)
        print("✓ Assignment completed successfully!")
        print("=" * 50)
        
    except Exception as e:
        print(f"\n❌ An error occurred: {e}")
        import traceback
        traceback.print_exc()

In [10]:
# ============================================
# CELL 7: Run the Program
# ============================================

# Execute the main function when notebook runs
if __name__ == "__main__":
    main()

NLP Assignment 1 - Sentence Generator (OPTIMIZED)
Team: 20221217_20220136_21088_20220040

[Step 1] Getting user input...
✓ Generating 8 sentences with N=3, maxLen=10

[Step 2] Loading and preprocessing Brown corpus...
Loading Brown corpus...
✓ Total tokens collected: 200100
✓ Total sentences collected: 10334
✓ Corpus loaded with 200100 tokens in 10334 sentences
✓ Vocabulary size: 21009 unique words

[Step 3] Building n-gram models...
  - Building unigram model...
  - Building bigram model...
  - Building trigram model...
✓ Unigram model: 21009 unique words
✓ Bigram model: 112815 unique bigrams
✓ Trigram model: 162712 unique trigrams

[Step 4] Precomputing probabilities for faster generation...
  - Precomputing bigram probabilities...
  - Precomputing trigram probabilities...
  - Organizing by context for O(1) lookup...
✓ Built 106613 context entries for trigram

[Step 5] Generating 8 sentences (optimized)...
✓ Generated 8 sentences in 0.01 seconds

[Step 6] Displaying generated sentenc

In [8]:
# ============================================
# CELL 8: Test Cases (Optional)
# ============================================

def test_with_small_corpus():
    """Test the entire pipeline with a small corpus"""
    print("=" * 40)
    print("Testing with small corpus...")
    print("=" * 40)
    
    # Create small test corpus
    test_corpus = [
        ["the", "cat", "sat", "on", "the", "mat"],
        ["the", "dog", "ran", "in", "the", "park"],
        ["the", "cat", "chased", "the", "mouse"],
        ["dogs", "bark", "loudly", "at", "night"],
        ["cats", "and", "dogs", "are", "pets"]
    ]
    
    print(f"Test corpus: {len(test_corpus)} sentences")
    
    # Get vocabulary
    vocabulary = set()
    for sent in test_corpus:
        vocabulary.update(sent)
    print(f"Vocabulary size: {len(vocabulary)} words")
    
    # Build models
    bigram_dict = build_bigram_model(test_corpus)
    trigram_dict = build_trigram_model(test_corpus)
    
    print(f"Bigram dict: {len(bigram_dict)} entries")
    print(f"Trigram dict: {len(trigram_dict)} entries")
    
    # Test probability calculations
    print("\n--- Testing Probability Calculations ---")
    prob = calculate_bigram_probability("cat", "the", bigram_dict)
    print(f"P(cat | the) = {prob:.3f}")
    
    prob = calculate_trigram_probability("sat", "the", "cat", trigram_dict)
    print(f"P(sat | the, cat) = {prob:.3f}")
    
    # Test top 3 selection
    print("\n--- Testing Top 3 Selection ---")
    context = ("the",)
    print(f"Context: {context}")
    word_probs = []
    for word in vocabulary:
        prob = calculate_bigram_probability(word, "the", bigram_dict)
        if prob > 0:
            word_probs.append((word, prob))
    
    word_probs.sort(key=lambda x: x[1], reverse=True)
    print("Top 3 words after 'the':")
    for i, (word, prob) in enumerate(word_probs[:3], 1):
        print(f"  {i}. '{word}' (prob: {prob:.3f})")
    
    # Test sentence generation with new top-3 selection
    print("\n--- Testing Sentence Generation (Top 3 Selection) ---")
    sentences = generate_sentences(5, 2, 6, bigram_dict, trigram_dict, test_corpus, vocabulary)
    display_sentences(sentences)
    
    print("\n✓ All tests passed!")
    return True

# Uncomment to run tests
# test_with_small_corpus()